In [28]:
import numpy as np
import torch
import os
import logging
import yaml
import sys
# script_dir = os.path.dirname(__file__) 
# helpers_path = os.path.join('oldver/NRAD/non-resonant-AD/model_scripts', 'model_scripts')  
helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/oldver/NRAD/non-resonant-AD/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
from SimpleMAF import SimpleMAF

In [22]:
seed = 2
data_path = f"SemiVisJets/data/data_seed{seed}"
mc_path = "SemiVisJets/data"
model_path = "models"
config_path = "oldver/NRAD/non-resonant-AD/Train_Models/configs"
os.makedirs(model_path, exist_ok=True)

In [23]:
print("Setting up device...")
CUDA = torch.cuda.is_available()
print("cuda available:", CUDA)
device = torch.device("cuda" if CUDA else "cpu")

Setting up device...
cuda available: True


In [24]:
print("Loadding MC events...")
mc_events = np.load(f"{mc_path}/mc_events_chunk{seed:02d}.npz", allow_pickle=True)
mc_events_cr = mc_events["mc_events_cr"]
mc_events_sr = mc_events["mc_events_sr"]

Loadding MC events...


In [25]:
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    # data_events_sr = data_chunk["data_events_sr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")

    n_context = 2

    data_cr_train = data_events_cr[:, :n_context]
    mc_cr_train = mc_events_cr[:, :n_context]

    input_x_train_CR = np.concatenate([mc_cr_train, data_cr_train], axis=0)

    mc_cr_label = np.zeros(mc_cr_train.shape[0]).reshape(-1, 1)
    data_cr_label = np.ones(data_cr_train.shape[0]).reshape(-1, 1)
    input_y_train_CR = np.concatenate([mc_cr_label, data_cr_label], axis=0)

    print("Training data shape:", input_x_train_CR.shape, input_y_train_CR.shape)

    with open(f"{config_path}/context_weights_physics.yml", 'r') as stream:
        params = yaml.safe_load(stream)

    NN_reweight = Classifier(n_inputs=n_context, layers=params["layers"], learning_rate=params["learning_rate"], device=device)
    print("Training context weights...")

    NN_reweight.train(input_x_train_CR, input_y_train_CR, save_model=True, batch_size=params["batch_size"], n_epochs=params["n_epochs"], model_name=f"context_weight_MC{seed:02d}_Data{i:02d}", outdir=model_path)

    print("Done training for data chunk", i)

Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Training data shape: (19936593, 2) (19936593, 1)
Training context weights...


 16%|=>        | 8/50 [20:43<1:48:49, 155.47s/it]


Done training for data chunk 1
Loading data chunk 2
CR has 9983941 data events, 9952849 MC events.
Training data shape: (19936790, 2) (19936790, 1)
Training context weights...


 36%|===>      | 18/50 [44:26<1:19:00, 148.13s/it]


Done training for data chunk 2
Loading data chunk 3
CR has 9983542 data events, 9952849 MC events.
Training data shape: (19936391, 2) (19936391, 1)
Training context weights...


...

Done training for data chunk 8
Loading data chunk 9
CR has 9983826 data events, 9952849 MC events.
Training data shape: (19936675, 2) (19936675, 1)
Training context weights...


 20%|==        | 10/50 [26:40<1:46:40, 160.02s/it]


Done training for data chunk 9
Loading data chunk 10
CR has 9983689 data events, 9952849 MC events.
Training data shape: (19936538, 2) (19936538, 1)
Training context weights...


 28%|==>       | 14/50 [35:29<1:31:16, 152.12s/it]

Done training for data chunk 10


In [ ]:
print("Training Reweight...")
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    # data_events_sr = data_chunk["data_events_sr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")
    # print("SR has", len(data_events_sr), "data events,", len(mc_events_sr), "MC events.")

    data_cr_train = data_events_cr
    mc_cr_train = mc_events_cr

    input_x_train_CR = np.concatenate([mc_cr_train, data_cr_train], axis=0)

    mc_cr_label = np.zeros(mc_cr_train.shape[0]).reshape(-1, 1)
    data_cr_label = np.ones(data_cr_train.shape[0]).reshape(-1, 1)
    input_y_train_CR = np.concatenate([mc_cr_label, data_cr_label], axis=0)

    print("Training data shape:", input_x_train_CR.shape, input_y_train_CR.shape)
    with open(f"{config_path}/reweight_physics.yml", 'r') as stream:
        params = yaml.safe_load(stream)


    NN_reweight = Classifier(n_inputs=7, layers=params["layers"], learning_rate=params["learning_rate"], device=device)
    print("Training reweight... at data chunk", i)

    NN_reweight.train(input_x_train_CR, input_y_train_CR, save_model=True, batch_size=params["batch_size"], n_epochs=params["n_epochs"], model_name=f"reweight_MC{seed:02d}_Data{i:02d}", outdir=model_path)
    print("Done training for data chunk", i)
print("All done!")



Training Reweight...
Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Training data shape: (19936593, 7) (19936593, 1)
Training reweight... at data chunk 1


 34%|===       | 17/50 [42:30<1:22:30, 150.01s/it]


Done training for data chunk 1
Loading data chunk 2
CR has 9983941 data events, 9952849 MC events.
Training data shape: (19936790, 7) (19936790, 1)
Training reweight... at data chunk 2


 32%|===       | 16/50 [40:12<1:25:25, 150.75s/it]


Done training for data chunk 2
Loading data chunk 3
CR has 9983542 data events, 9952849 MC events.
Training data shape: (19936391, 7) (19936391, 1)
Training reweight... at data chunk 3


...

Done training for data chunk 8
Loading data chunk 9
CR has 9983826 data events, 9952849 MC events.
Training data shape: (19936675, 7) (19936675, 1)
Training reweight... at data chunk 9


 24%|==        | 12/50 [31:01<1:38:13, 155.09s/it]


Done training for data chunk 9
Loading data chunk 10
CR has 9983689 data events, 9952849 MC events.
Training data shape: (19936538, 7) (19936538, 1)
Training reweight... at data chunk 10


 20%|==        | 10/50 [25:58<1:43:55, 155.89s/it]

Done training for data chunk 10
All done!


In [ ]:
print("Training Generate...")
for i in range(1, 11):
    print("Loading data chunk", i)
    data_chunk = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    data_events_cr = data_chunk["data_events_cr"]
    # data_events_sr = data_chunk["data_events_sr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")
    # print("SR has", len(data_events_sr), "data events,", len(mc_events_sr), "MC events.")

    n_context = 2
    n_features = 5

    data_context_cr_train = data_events_cr[:, :n_context]
    data_feature_cr_train = data_events_cr[:, n_context:]

    mc_context_sr = mc_events_sr[:, :n_context]
    mc_feature_sr = mc_events_sr[:, n_context:]

    with open(f"{config_path}/generate_physics.yml", 'r') as stream:
        params = yaml.safe_load(stream)

    MAF = SimpleMAF(num_features=n_features, num_context=n_context, device=device, num_layers=params["n_layers"], num_hidden_features=params["n_hidden_features"], learning_rate = params["learning_rate"])
    print("Training Generate... at data chunk", i)
    
    MAF.train(data=data_feature_cr_train, cond=data_context_cr_train, batch_size=params["batch_size"], n_epochs=params["n_epochs"], outdir=model_path, save_model=True, model_name=f"generate_MC{seed:02d}_Data{i:02d}")
    print("Done training for data chunk", i)
print("All done!")

Training Generate...
Loading data chunk 1
CR has 9983744 data events, 9952849 MC events.
Training Generate... at data chunk 1


 65%|======>   | 13/20 [2:03:13<1:06:21, 568.75s/it]


Done training for data chunk 1
Loading data chunk 2
CR has 9983941 data events, 9952849 MC events.
Training Generate... at data chunk 2


 55%|=====>    | 11/20 [1:45:02<1:25:56, 573.00s/it]


Done training for data chunk 2
Loading data chunk 3
CR has 9983542 data events, 9952849 MC events.
Training Generate... at data chunk 3


 30%|===       | 6/20 [51:09<2:00:15, 515.39s/it]